# Plot a saved simulated outcome

First run the “Distribution of one simulated outcome” cell in `roc_simulation_fixed_lambda1.ipynb` to save a `.npy` file. This notebook loads those saved y values without refitting the model or regenerating the simulation. Old result folders that did not save y cannot be loaded here.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

MODEL_DIR = Path.cwd() / 'model'
if not (MODEL_DIR / 'roc_experiment.py').exists():
    MODEL_DIR = Path.cwd()
RESULTS_DIR = MODEL_DIR / 'results'

saved_y_files = sorted(
    RESULTS_DIR.glob('*/figures/simulated_y_target_*_rep_*.npy'),
    key=lambda path: path.stat().st_mtime, reverse=True,
)
print(f'Found {len(saved_y_files)} saved outcome file(s):')
for path in saved_y_files:
    print(path)


## Choose and plot an outcome

Leave `Y_FILE` as `None` to use the most recently saved file, or set it to the full path of a file listed above.


In [ ]:
Y_FILE = None  # Example: '/full/path/to/simulated_y_target_0.7_rep_1.npy'

if Y_FILE is None:
    if not saved_y_files:
        raise FileNotFoundError(
            'No saved y values found. Run the simulated-y distribution cell in '
            'roc_simulation_fixed_lambda1.ipynb first.'
        )
    y_path = saved_y_files[0]
else:
    y_path = Path(Y_FILE).expanduser()

simulated_y = np.load(y_path, allow_pickle=False).ravel()
if simulated_y.size == 0 or not np.isfinite(simulated_y).all():
    raise ValueError('Saved y must contain finite values.')

fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
ax.hist(simulated_y, bins='auto', color='steelblue', edgecolor='white')
ax.axvline(np.mean(simulated_y), color='darkorange', linestyle='--',
           label=f'Mean = {np.mean(simulated_y):.2f}')
ax.set(xlabel='Simulated y (before centering)', ylabel='Number of subjects',
       title=y_path.stem.replace('_', ' '))
ax.legend()
print(f'Loaded: {y_path}')
print(f'n={len(simulated_y)}, mean={np.mean(simulated_y):.3f}, '
      f'sd={np.std(simulated_y, ddof=1):.3f}')
plt.show()
